# Tile encoder incelemesi

Bu notebook kaydedilmiş oyunlardan rastgele bir observation seçer ve şu ana kadarki akışı gösterir:

`raw tile -> Tile nesnesi -> kategorik ID + numeric özellik -> embedding ile birleştirme -> 1x1 Conv`

`1x1 Conv` her tile'ı kendi içinde işler; komşu tile'lar henüz karışmaz.

In [1]:
import json
import random
from pathlib import Path

import torch

from global_encoding import GLOBAL_FEATURE_NAMES, encode_global_observation
from kinds import parse_tile_grid
from model import GlobalEncoder, StateEncoder, TileEncoder
from observation_encoding import encode_observation
from tile_encoding import (
    DYNAMIC_FEATURE_NAMES,
    NUMERIC_FEATURE_NAMES,
    STATIC_FEATURE_NAMES,
    encode_tile_grid,
)

## 1. Rastgele observation seç

Aşağıdaki hücreyi yeniden çalıştırarak farklı oyun, adım, oyuncu ve tarla seçebilirsin.

In [2]:
game_paths = sorted(Path("game_observations").glob("game_*.json"))
if not game_paths:
    raise FileNotFoundError("game_observations altında oyun JSON'u bulunamadı")

game_path = random.choice(game_paths)
with game_path.open(encoding="utf-8") as file:
    game = json.load(file)

required_observation_keys = {"step", "day", "hour", "farms", "market"}
valid_observations = [
    (step_record, player)
    for step_record in game["steps"]
    for player in step_record["players"]
    if required_observation_keys <= set(player.get("observation") or {})
]
step_record, player_record = random.choice(valid_observations)
observation = player_record["observation"]
farm_index = random.randrange(len(observation["farms"]))
farm = observation["farms"][farm_index]

print("Dosya:      ", game_path.name)
print("Step:       ", observation["step"])
print("Day / hour: ", observation["day"], observation["hour"])
print("Player:     ", observation["player"])
print("Farm index: ", farm_index)

Dosya:       game_04_seed_404.json
Step:        189
Day / hour:  7 21
Player:      0
Farm index:  0


## 2. Ham tile gridini parse ve encode et

- `categorical_ids`: kind, crop ve animal ID kanallarıdır.
- `numeric_features`: dinamik tile durumu ile crop/animal türünün sabit oyun kurallarını içerir.

In [3]:
raw_tiles = farm["tiles"]
parsed_tiles = parse_tile_grid(raw_tiles)

categorical_ids, numeric_features = encode_tile_grid(
    tile_grid=parsed_tiles,
    current_day=observation["day"],
    current_step=observation["step"],
    market_prices=observation["market"]["prices"],
)

print("categorical_ids: ", categorical_ids.shape, categorical_ids.dtype)
print("numeric_features:", numeric_features.shape, numeric_features.dtype)
print("dinamik kanallar: ", len(DYNAMIC_FEATURE_NAMES))
print("statik kanallar:  ", len(STATIC_FEATURE_NAMES))
print("numeric toplam:   ", len(NUMERIC_FEATURE_NAMES))

categorical_ids:  (3, 10, 10) int64
numeric_features: (11, 10, 10) float32
numeric kanallar:  11


## 3. Batch boyutu ekle

Model girdisi batch ile çalışır. Tek tarlamız olduğu için `B=1`.

In [4]:
categorical_tensor = torch.from_numpy(categorical_ids).long().unsqueeze(0)
numeric_tensor = torch.from_numpy(numeric_features).float().unsqueeze(0)

print("categorical_tensor:", categorical_tensor.shape)
print("numeric_tensor:    ", numeric_tensor.shape)

categorical_tensor: torch.Size([1, 3, 10, 10])
numeric_tensor:     torch.Size([1, 11, 10, 10])


## 4. Embedding ile numeric kanalları aynı tile'da birleştir

Embedding toplam 11 kanaldır. Numeric taraf 12 dinamik ve 14 statik olmak üzere 26 kanaldır. Aynı `(y, x)` koordinatında birleştirilince her tile 37 kanala sahip olur. `TileEncoder` daha sonra ortak 37 değeri `1x1 Conv` ile 32 değere dönüştürür.

In [5]:
torch.manual_seed(42)
tile_encoder = TileEncoder(output_channels=32)

kind_ids = categorical_tensor[:, 0]
crop_ids = categorical_tensor[:, 1]
animal_ids = categorical_tensor[:, 2]

embedded_channels_last = tile_encoder.embedding(
    kind_ids, crop_ids, animal_ids
)
embedded_channels_first = embedded_channels_last.permute(0, 3, 1, 2)
combined_features = tile_encoder.combine_features(
    categorical_tensor, numeric_tensor
)
encoded_tiles = tile_encoder(categorical_tensor, numeric_tensor)

print("Embedding (channels last): ", embedded_channels_last.shape)
print("Embedding (channels first):", embedded_channels_first.shape)
print("Numeric:                     ", numeric_tensor.shape)
print("Birleşik:                    ", combined_features.shape)
print("1x1 Conv çıktısı:           ", encoded_tiles.shape)

Embedding (channels last):  torch.Size([1, 10, 10, 11])
Embedding (channels first): torch.Size([1, 11, 10, 10])
Numeric:                      torch.Size([1, 11, 10, 10])
Birleşik:                     torch.Size([1, 22, 10, 10])
1x1 Conv çıktısı:            torch.Size([1, 32, 10, 10])


Boyut akışı:

```text
categorical IDs  (B,  3, 10, 10)
       -> embedding
embedding        (B, 11, 10, 10) --+
numeric          (B, 26, 10, 10) --+-> concat -> (B, 37, 10, 10)
                                             -> 1x1 Conv -> (B, 32, 10, 10)
```

## 5. Aynı koordinattaki tek tile'ı adım adım incele

Varsa bitki veya hayvan içeren bir tile tercih edilir. Yoksa dictionary tile, o da yoksa rastgele bir tile seçilir.

In [6]:
height = len(raw_tiles)
width = len(raw_tiles[0])
all_coordinates = [(y, x) for y in range(height) for x in range(width)]
dynamic_coordinates = [
    (y, x)
    for y, x in all_coordinates
    if isinstance(raw_tiles[y][x], dict)
    and (raw_tiles[y][x].get("kind") == "PLANT" or "animal" in raw_tiles[y][x])
]
dictionary_coordinates = [
    (y, x)
    for y, x in all_coordinates
    if isinstance(raw_tiles[y][x], dict)
]
candidate_coordinates = dynamic_coordinates or dictionary_coordinates or all_coordinates
y, x = random.choice(candidate_coordinates)

tile_embedding = embedded_channels_first[0, :, y, x]
tile_numeric = numeric_tensor[0, :, y, x]
tile_combined = combined_features[0, :, y, x]
tile_encoded = encoded_tiles[0, :, y, x]
expected_combination = torch.cat([tile_embedding, tile_numeric])

print(f"Koordinat: (y={y}, x={x})")
print("Ham tile:      ", raw_tiles[y][x])
print("Parsed tile:   ", parsed_tiles[y][x])
print("Kategorik ID:  ", categorical_tensor[0, :, y, x].tolist())
print("Embedding:     ", tile_embedding.shape, tile_embedding)
print("Numeric:       ", tile_numeric.shape)
print(dict(zip(NUMERIC_FEATURE_NAMES, tile_numeric.tolist())))
print("Birleşik:      ", tile_combined.shape, tile_combined)
print("1x1 Conv çıktı:", tile_encoded.shape, tile_encoded)
print("Koordinat eşleşmesi doğru mu?:", torch.allclose(tile_combined, expected_combination))

Koordinat: (y=2, x=2)
Ham tile:       {'kind': 'WEED'}
Parsed tile:    WeedTile()
Kategorik ID:   [2, 0, 0]
Embedding:      torch.Size([11]) tensor([ 0.3559, -0.6866, -0.4934,  0.2415,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000], grad_fn=<SelectBackward0>)
Numeric:        torch.Size([11])
{'age': 0.0, 'basic_need_done_today': 0.0, 'neglect_streak': 0.0, 'yield_fill': 0.0, 'lifespan_known': 0.0, 'lifespan_remaining': 0.0, 'plant_fertilizer_remaining': 0.0, 'animal_cared_today': 0.0, 'animal_fertilizer_available': 0.0, 'animal_pending_care_bonus': 0.0, 'animal_occupied': 0.0}
Birleşik:       torch.Size([22]) tensor([ 0.3559, -0.6866, -0.4934,  0.2415,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
       grad_fn=<SelectBackward0>)
1x1 Conv çıktı: torch.Size([32]) tensor([0.3472, 0.0000, 0.0317, 0.0000, 0.0000, 0.0000, 0.2696, 0.000

## Sonuç

- Embedding ve numeric bilgi farklı tile'lar arasında değil, aynı `(y, x)` koordinatında birleşir.
- `1x1 Conv`, tek tile'ın kategori ve durum bilgilerini birlikte işler.
- Çıktıdaki `(B, 32, 10, 10)` gridin konumları hâlâ korunur.
- Bu notebook `TileEncoder` çıktısında durur; `model.py` içindeki `4x4 SpatialEncoder` sonraki adımda komşu tile'ları birbirine bağlar.

## 6. Global observation kolu

Bu kol grid dışındaki zaman, market, ekonomi, envanter ve town bilgilerini 62 elemanlı sabit bir vektöre çevirir. MLP bu vektörü 64 boyutlu öğrenilmiş temsile dönüştürür.

In [ ]:
global_features = encode_global_observation(observation)
global_tensor = torch.from_numpy(global_features).float().unsqueeze(0)

global_encoder = GlobalEncoder(output_dim=64)
global_encoded = global_encoder(global_tensor)

print("Ham global vektör:", global_tensor.shape)
print("Global MLP çıktısı:", global_encoded.shape)

In [ ]:
global_values = dict(zip(GLOBAL_FEATURE_NAMES, global_features.tolist()))

for name in (
    "season_progress",
    "market_price_WHEAT",
    "market_price_MELON",
    "market_inventory_WHEAT",
    "own_money",
    "opponent_money",
    "seed_WHEAT",
    "shed_FERTILIZER",
):
    print(f"{name}: {global_values[name]:.4f}")

## 7. Own farm, opponent farm ve global fusion

Own ve opponent farm aynı `FarmEncoder` nesnesinden geçer. Policy için own spatial map korunur; opponent ve global özetler her own tile'a yayılır. Value tarafı için 256 boyutlu state vektörü üretilir.

In [ ]:
encoded_observation = encode_observation(observation)

def to_batch(name, dtype):
    array = getattr(encoded_observation, name)
    return torch.from_numpy(array).to(dtype).unsqueeze(0)

state_encoder = StateEncoder()
state = state_encoder(
    own_categorical_ids=to_batch("own_categorical_ids", torch.long),
    own_numeric_features=to_batch("own_numeric_features", torch.float32),
    own_worker_positions=to_batch("own_worker_positions", torch.float32),
    opponent_categorical_ids=to_batch("opponent_categorical_ids", torch.long),
    opponent_numeric_features=to_batch("opponent_numeric_features", torch.float32),
    opponent_worker_positions=to_batch("opponent_worker_positions", torch.float32),
    global_features=to_batch("global_features", torch.float32),
)

print("Own map:         ", state.own_map.shape)
print("Opponent map:    ", state.opponent_map.shape)
print("Own summary:     ", state.own_summary.shape)
print("Opponent summary:", state.opponent_summary.shape)
print("Global summary:  ", state.global_summary.shape)
print("Policy features: ", state.policy_features.shape)
print("State vector:    ", state.state_vector.shape)